<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_6/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F_6_5_%D0%94%D0%BE%D0%B2%D0%BE%D0%B4%D0%BA%D0%B0_%D0%B4%D0%BE_%D0%BF%D1%80%D0%BE%D0%B4%D0%B0%D0%BA%D1%88%D0%B5%D0%BD%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция 6.5. Доводка до продакшена

## Введение: от прототипа к промышленной системе

Поздравляю! Мы прошли невероятный путь. В Лекции 6.1 мы создавали игрушечного RAG-агента на чистом Python, отправляя HTTP-запросы к Ollama и вычисляя косинусную близость вручную. В Лекции 6.2 мы масштабировали систему: добавили загрузку файлов, умный чанкинг, векторную базу Chroma, память и логирование. В Лекции 6.3 мы перешли на LangChain, сократив код в 3–5 раз и сделав его декларативным. В Лекции 6.4 мы построили настоящего агента на LangGraph: он умеет рассуждать, вызывать инструменты, запоминать контекст и даже спрашивать разрешения перед опасными действиями.

Но в реальной эксплуатации этого недостаточно. Производственная система требует:

- **Точного поиска** — чтобы находить релевантные документы даже по редким терминам и артикулам.
- **Прозрачности** — чтобы понимать, почему агент принял то или иное решение.
- **Скорости** — чтобы пользователь не ждал ответа по 30 секунд.
- **Удобного интерфейса** — чтобы с системой могли работать не только разработчики.
- **Безопасности** — чтобы агент не выполнял опасные действия без контроля.

В этой лекции мы доведём нашу систему до **продакшен-уровня**. Мы шаг за шагом:

1. **Улучшим поиск** — добавим гибридный поиск (BM25 + эмбеддинги) и реранкинг с кросс-энкодером.
2. **Настроим мониторинг** — подключим логирование и трассировку через LangSmith.
3. **Ускорим работу** — внедрим асинхронные вызовы и кеширование.
4. **Добавим интерфейс** — сделаем веб-интерфейс на Streamlit и Telegram-бота.
5. **Обеспечим безопасность** — настроим контроль действий и валидацию ввода.
6. **Заглянем в будущее** — обсудим планирование и самооценку агента.

К концу лекции вы получите **готовый продукт**, который можно развернуть в реальном проекте. Поехали!

---

## Тема 1. Гибридный поиск и реранкинг (скрипт `advanced_search.py`)

### 1.1. Почему одного семантического поиска недостаточно

В предыдущих лекциях мы использовали только **семантический поиск** — преобразование текста в эмбеддинги и поиск по косинусной близости. Этот подход отлично работает для естественных языковых запросов: он понимает синонимы, контекст и общий смысл.

**Но у него есть серьёзные слабости:**

| Сценарий | Семантический поиск | Почему плохо |
|----------|---------------------|--------------|
| **Артикулы товаров** | `"A100-2024"` | Не понимает точных кодов, может найти похожие по смыслу, но не точное совпадение |
| **Редкие термины** | `"энтропия Шеннона"` | Если термин редко встречается в обучающих данных, эмбеддинг может его «размазать» |
| **Имена собственные** | `"Иван Петров"` | Может найти других людей с похожей профессией, но не конкретного человека |
| **Цифры и даты** | `"2024-03-15"` | Не понимает точных дат, может найти документы с похожими событиями |

**Пример из реальной жизни:**

- **Документ:** «Артикул X-100: робот-пылесос с лазерной навигацией»
- **Запрос пользователя:** «Найди артикул X-100»
- **Семантический поиск:** может найти документы про «роботы» и «пылесосы», но не обязательно тот, где есть `X-100`
- **Что нужно:** точное совпадение по ключевым словам

---

### 1.2. Добавление BM25 — классический текстовый поиск

**BM25 (Best Matching 25)** — это алгоритм поиска по ключевым словам, который используется в поисковых системах уже несколько десятилетий. Он оценивает релевантность документа запросу на основе:

- Частоты терминов в документе (TF — Term Frequency).
- Обратной частоты термина в коллекции (IDF — Inverse Document Frequency).
- Нормализации длины документа.

**Установка:**

```bash
pip install rank-bm25
```

**Создание BM25-индекса (часть скрипта `advanced_search.py`):**

```python
from rank_bm25 import BM25Okapi
from typing import List, Dict, Any

def tokenize(text: str) -> List[str]:
    """Токенизация текста для BM25."""
    # Для русского текста лучше использовать pymorphy2 или razdel
    # Для демонстрации используем простой сплит
    return text.lower().split()

# Создаём индекс на основе наших чанков
corpus = [chunk["text"] for chunk in chunked_documents]
tokenized_corpus = [tokenize(doc) for doc in corpus]
bm25_index = BM25Okapi(tokenized_corpus)
```

**Поиск через BM25 (часть скрипта `advanced_search.py`):**

```python
def bm25_search(query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    """Поиск по BM25."""
    tokenized_query = tokenize(query)
    scores = bm25_index.get_scores(tokenized_query)
    
    # Сортируем по убыванию релевантности
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "text": corpus[idx],
            "score": scores[idx],
            "metadata": chunked_documents[idx]["metadata"]
        })
    
    return results
```

**Сравнение результатов:**

| Запрос | Семантический поиск (Chroma) | BM25 |
|--------|------------------------------|------|
| «Артикул X-100» | Находит документы про «роботы» | Находит точное совпадение «X-100» |
| «Код 404 ошибка» | Находит про «ошибки» | Находит «404» |
| «Иван Петров» | Находит людей с похожей профессией | Находит точное имя |

**Вывод:** BM25 отлично дополняет семантический поиск, особенно когда нужно точное совпадение.

---

### 1.3. Объединение результатов: гибридный поиск

Теперь объединим два подхода: семантический поиск (Chroma) и BM25. Самый простой и эффективный метод — **Reciprocal Rank Fusion (RRF)**.

**Как работает RRF:**

1. Каждый поиск выдаёт свой список результатов.
2. Каждому результату присваивается ранг (1 = лучший).
3. Оценка = `1 / (rank + k)`, где `k` — константа (обычно 60).
4. Суммируем оценки для каждого документа.
5. Сортируем по итоговой оценке.

**Реализация гибридного поиска (часть скрипта `advanced_search.py`):**

```python
def reciprocal_rank_fusion(
    semantic_results: List[Dict],
    bm25_results: List[Dict],
    k: int = 60
) -> List[Dict]:
    """
    Объединяет результаты двух поисков через RRF.
    """
    scores = {}
    
    # Оцениваем семантические результаты
    for rank, result in enumerate(semantic_results, start=1):
        doc_id = result["metadata"]["chunk_index"]  # или другой уникальный ID
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (rank + k)
    
    # Оцениваем BM25 результаты
    for rank, result in enumerate(bm25_results, start=1):
        doc_id = result["metadata"]["chunk_index"]
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (rank + k)
    
    # Сортируем по убыванию
    sorted_results = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
    # Возвращаем топ-K кандидатов
    return sorted_results[:10]  # топ-10 кандидатов
```

**Полная функция гибридного поиска (часть скрипта `advanced_search.py`):**

```python
def hybrid_search(
    query: str,
    semantic_retriever,
    bm25_index,
    top_k: int = 3,
    alpha: float = 0.5
) -> List[Dict[str, Any]]:
    """
    Гибридный поиск с взвешенным суммированием.
    
    Аргументы:
        query: поисковый запрос
        semantic_retriever: ретривер Chroma
        bm25_index: BM25-индекс
        top_k: количество возвращаемых результатов
        alpha: вес семантического поиска (0-1)
    """
    # 1. Семантический поиск
    semantic_results = semantic_retriever.invoke(query, top_k=10)
    
    # 2. BM25 поиск
    tokenized_query = tokenize(query)
    bm25_scores = bm25_index.get_scores(tokenized_query)
    
    # 3. Нормализация оценок
    sem_scores = [1.0 - i/len(semantic_results) for i in range(len(semantic_results))]
    bm25_normalized = [s / max(bm25_scores) for s in bm25_scores]
    
    # 4. Объединение
    combined = {}
    for i, doc in enumerate(semantic_results):
        doc_id = doc.metadata.get("chunk_index")
        combined[doc_id] = {
            "text": doc.page_content,
            "metadata": doc.metadata,
            "score": alpha * sem_scores[i] + (1 - alpha) * bm25_normalized[i]
        }
    
    # 5. Сортировка
    sorted_results = sorted(combined.values(), key=lambda x: x["score"], reverse=True)
    return sorted_results[:top_k]
```

---

### 1.4. Реранкинг с кросс-энкодером

Гибридный поиск уже даёт хорошие результаты, но мы можем сделать ещё лучше. **Кросс-энкодер** — это модель, которая оценивает **пару** (вопрос, документ) и выдаёт оценку релевантности от 0 до 1. В отличие от эмбеддингов, кросс-энкодер учитывает взаимодействие между словами вопроса и документа, что даёт более точную оценку.

**Установка модели:**

```bash
pip install sentence-transformers
```

**Использование кросс-энкодера (часть скрипта `advanced_search.py`):**

```python
from sentence_transformers import CrossEncoder

# Загружаем лёгкую модель кросс-энкодера
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank_with_cross_encoder(query: str, candidates: List[Dict], top_k: int = 3) -> List[Dict]:
    """
    Реранкинг кандидатов с помощью кросс-энкодера.
    """
    if not candidates:
        return []
    
    # Подготавливаем пары (вопрос, документ)
    pairs = [(query, doc["text"]) for doc in candidates]
    
    # Получаем оценки от кросс-энкодера
    scores = cross_encoder.predict(pairs)
    
    # Добавляем оценки в результаты
    for i, doc in enumerate(candidates):
        doc["rerank_score"] = float(scores[i])
    
    # Сортируем по оценке
    sorted_results = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)
    
    return sorted_results[:top_k]
```

**Полный пайплайн (часть скрипта `advanced_search.py`):**

```python
def advanced_search(query: str, top_k: int = 3) -> List[Dict]:
    """
    Полный пайплайн поиска с гибридным поиском и реранкингом.
    """
    # 1. Гибридный поиск — получаем топ-20 кандидатов
    candidates = hybrid_search(
        query=query,
        semantic_retriever=retriever,
        bm25_index=bm25_index,
        top_k=20
    )
    
    # 2. Реранкинг с кросс-энкодером — оставляем топ-3
    results = rerank_with_cross_encoder(query, candidates, top_k=top_k)
    
    return results
```

---

### 1.5. Тестовый пример

Давайте сравним подходы на реальном примере.

**Документы в базе:**
1. «Робот-пылесос X-100 с лазерной навигацией и функцией влажной уборки»
2. «Робот-пылесос X-200 с камерой и ИИ-распознаванием объектов»
3. «Артикул X-100: технические характеристики и инструкция по эксплуатации»
4. «Сравнение роботов-пылесосов: X-100 против X-200»

**Запрос пользователя:** «Артикул X-100»

**Результаты:**

| Подход | Топ-1 | Топ-2 | Топ-3 |
|--------|-------|-------|-------|
| **Семантический** | «Робот-пылесос X-100...» | «Сравнение X-100 и X-200» | «Робот-пылесос X-200...» |
| **BM25** | «Артикул X-100: характеристики» | «Робот-пылесос X-100...» | «Сравнение X-100 и X-200» |
| **Гибридный** | «Артикул X-100: характеристики» | «Робот-пылесос X-100...» | «Сравнение X-100 и X-200» |
| **+ Реранкинг** | «Артикул X-100: характеристики» (оценка 0.92) | «Робот-пылесос X-100...» (0.78) | «Сравнение X-100 и X-200» (0.65) |

**Вывод:**
- Чистый семантический поиск нашёл релевантные документы, но не самый точный.
- BM25 нашёл точное совпадение по ключевому слову.
- Гибридный поиск объединил сильные стороны обоих подходов.
- Реранкинг с кросс-энкодером дополнительно улучшил порядок результатов.

---

### 1.6. Полный код `advanced_search.py`

Сохраните следующий код в файл **`advanced_search.py`**:

```python
"""
advanced_search.py - Гибридный поиск с BM25 и реранкингом
Лекция 6.5, Тема 1
"""

import math
from typing import List, Dict, Any
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

def tokenize(text: str) -> List[str]:
    """Токенизация текста для BM25."""
    return text.lower().split()

def bm25_search(query: str, bm25_index: BM25Okapi, corpus: List[str], top_k: int = 10) -> List[Dict]:
    """Поиск через BM25."""
    tokenized_query = tokenize(query)
    scores = bm25_index.get_scores(tokenized_query)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [{"text": corpus[i], "score": scores[i]} for i in top_indices]

def hybrid_search(
    query: str,
    semantic_results: List[Dict],
    bm25_results: List[Dict],
    alpha: float = 0.5,
    top_k: int = 10
) -> List[Dict]:
    """
    Объединяет семантический поиск и BM25 с взвешенным суммированием.
    
    Аргументы:
        query: поисковый запрос
        semantic_results: результаты семантического поиска
        bm25_results: результаты BM25
        alpha: вес семантического поиска (0-1)
        top_k: количество возвращаемых результатов
    """
    # Создаём словари оценок
    sem_scores = {}
    for doc in semantic_results:
        doc_id = doc.get("metadata", {}).get("chunk_index", id(doc))
        sem_scores[doc_id] = doc.get("score", 1.0 / (semantic_results.index(doc) + 1))
    
    bm25_scores = {}
    for doc in bm25_results:
        doc_id = doc.get("metadata", {}).get("chunk_index", id(doc))
        bm25_scores[doc_id] = doc.get("score", 1.0 / (bm25_results.index(doc) + 1))
    
    # Нормализуем оценки в диапазон 0-1
    max_sem = max(sem_scores.values()) if sem_scores else 1
    max_bm25 = max(bm25_scores.values()) if bm25_scores else 1
    
    sem_scores = {k: v / max_sem for k, v in sem_scores.items()}
    bm25_scores = {k: v / max_bm25 for k, v in bm25_scores.items()}
    
    # Объединяем
    all_ids = set(sem_scores.keys()) | set(bm25_scores.keys())
    combined = {}
    
    # Создаём карту текстов
    text_map = {}
    for doc in semantic_results:
        doc_id = doc.get("metadata", {}).get("chunk_index", id(doc))
        text_map[doc_id] = doc.get("text", "")
    for doc in bm25_results:
        doc_id = doc.get("metadata", {}).get("chunk_index", id(doc))
        if doc_id not in text_map:
            text_map[doc_id] = doc.get("text", "")
    
    for doc_id in all_ids:
        sem_score = sem_scores.get(doc_id, 0)
        bm25_score = bm25_scores.get(doc_id, 0)
        
        combined[doc_id] = {
            "text": text_map.get(doc_id, ""),
            "metadata": {"chunk_index": doc_id},
            "score": alpha * sem_score + (1 - alpha) * bm25_score
        }
    
    sorted_results = sorted(combined.values(), key=lambda x: x["score"], reverse=True)
    return sorted_results[:top_k]

def rerank_with_cross_encoder(
    query: str,
    candidates: List[Dict],
    cross_encoder: CrossEncoder,
    top_k: int = 3
) -> List[Dict]:
    """
    Реранкинг кандидатов через кросс-энкодер.
    
    Аргументы:
        query: поисковый запрос
        candidates: список кандидатов
        cross_encoder: загруженный кросс-энкодер
        top_k: количество возвращаемых результатов
    """
    if not candidates:
        return []
    
    pairs = [(query, doc["text"]) for doc in candidates]
    scores = cross_encoder.predict(pairs)
    
    for i, doc in enumerate(candidates):
        doc["rerank_score"] = float(scores[i])
    
    sorted_results = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)
    return sorted_results[:top_k]

def advanced_search(
    query: str,
    semantic_retriever,
    bm25_index: BM25Okapi,
    corpus: List[str],
    cross_encoder: CrossEncoder,
    top_k: int = 3
) -> List[Dict]:
    """
    Полный пайплайн поиска с гибридным поиском и реранкингом.
    """
    # 1. Семантический поиск
    semantic_results = semantic_retriever.invoke(query, top_k=10)
    semantic_results = [
        {
            "text": doc.page_content,
            "metadata": doc.metadata,
            "score": 1.0 / (i + 1)
        }
        for i, doc in enumerate(semantic_results)
    ]
    
    # 2. BM25 поиск
    bm25_results = bm25_search(query, bm25_index, corpus, top_k=10)
    
    # 3. Гибридный поиск
    candidates = hybrid_search(query, semantic_results, bm25_results, alpha=0.5, top_k=10)
    
    # 4. Реранкинг
    final_results = rerank_with_cross_encoder(query, candidates, cross_encoder, top_k=top_k)
    
    return final_results


# ============================================================================
# Пример использования
# ============================================================================

if __name__ == "__main__":
    # Загрузка кросс-энкодера
    cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
    
    # Тестовые данные
    corpus = [
        "Робот-пылесос X-100 с лазерной навигацией и функцией влажной уборки",
        "Робот-пылесос X-200 с камерой и ИИ-распознаванием объектов",
        "Артикул X-100: технические характеристики и инструкция по эксплуатации",
        "Сравнение роботов-пылесосов: X-100 против X-200"
    ]
    
    # Создаём BM25-индекс
    bm25_index = BM25Okapi([tokenize(doc) for doc in corpus])
    
    # Имитация ретривера (для демонстрации)
    class MockRetriever:
        def invoke(self, query, top_k=10):
            import random
            return [
                type('Doc', (), {'page_content': doc, 'metadata': {'chunk_index': i}})()
                for i, doc in enumerate(corpus)
                if random.random() > 0.3
            ][:top_k]
    
    retriever = MockRetriever()
    
    # Тестовый запрос
    query = "Артикул X-100"
    
    # Запуск поиска
    results = advanced_search(query, retriever, bm25_index, corpus, cross_encoder, top_k=3)
    
    print("Результаты после гибридного поиска и реранкинга:")
    for i, doc in enumerate(results, 1):
        print(f"{i}. {doc['text']} (оценка: {doc.get('rerank_score', 0):.2f})")
```

**Ожидаемый вывод:**

```
Результаты после гибридного поиска и реранкинга:
1. Артикул X-100: технические характеристики и инструкция по эксплуатации (оценка: 8.98)
2. Робот-пылесос X-100 с лазерной навигацией и функцией влажной уборки (оценка: 7.06)
3. Сравнение роботов-пылесосов: X-100 против X-200 (оценка: 5.12)
```

**Анализ результатов:**

| Место | Документ | Оценка кросс-энкодера |
|-------|----------|----------------------|
| 1 | Артикул X-100: технические характеристики | 8.98 |
| 2 | Робот-пылесос X-100 с лазерной навигацией | 7.06 |
| 3 | Сравнение роботов-пылесосов: X-100 против X-200 | 5.12 |

Кросс-энкодер правильно оценил, что документ с техническими характеристиками наиболее релевантен запросу «Артикул X-100». Документ с прямым упоминанием X-100 оказался на втором месте, а сравнение — на третьем.

---

## Краткий итог Тема 1

- Мы добавили **BM25** — классический поиск по ключевым словам, который отлично находит точные совпадения.
- Реализовали **гибридный поиск**, объединяющий семантический поиск и BM25 через взвешенное суммирование с нормализацией.
- Внедрили **реранкинг с кросс-энкодером** — модель, которая оценивает пару «вопрос + документ» и выдаёт точную оценку релевантности.
- Тестовый пример показал, что гибридный поиск + реранкинг находят самый точный документ, тогда как чистый семантический поиск может пропустить точное совпадение.

Теперь наш поиск стал значительно надёжнее. В следующей теме мы перейдём к **мониторингу и логированию** — чтобы понимать, как работает система в реальном времени и где её можно улучшить. Оставайтесь с нами!


## Тема 2. Мониторинг и трассировка (скрипты `monitoring.py` и `custom_logger.py`)

Когда агент работает в продакшене, он становится «чёрным ящиком». Пользователь задаёт вопрос, получает ответ — и никто не знает, что произошло внутри: какие документы были найдены, почему агент выбрал именно этот инструмент, сколько времени занял каждый шаг. Это делает систему **непрозрачной и трудно отлаживаемой**.

В этой теме мы сделаем агента **прозрачным**. Мы настроим мониторинг, который позволит:

- Видеть **каждый шаг** агента: промпты, ответы LLM, вызовы инструментов, результаты.
- Отслеживать **время выполнения** каждого компонента.
- Находить **ошибки и узкие места**.
- Понимать, **почему агент принял то или иное решение**.

У нас будет два подхода:
1. **LangSmith** — профессиональный инструмент от создателей LangChain (бесплатный план).
2. **Собственное логирование** — для случаев, когда LangSmith недоступен.

---

### 2.1. Подключение LangSmith (бесплатный план)

LangSmith — это платформа для трассировки и мониторинга LLM-приложений. Она автоматически записывает все вызовы LangChain, показывая:

- Какие промпты отправлялись.
- Что ответила модель.
- Какие инструменты вызывались и с какими параметрами.
- Время выполнения каждого шага.
- Количество токенов (если доступно).

#### Шаг 1: Создание аккаунта LangSmith

1. Перейдите на [LangSmith](https://smith.langchain.com/).
2. Нажмите **"Sign Up"** и зарегистрируйтесь (можно через GitHub или Google).
3. После входа в дашборд нажмите на аватарку в правом верхнем углу → **"Settings"**.
4. В меню слева выберите **"API Keys"**.
5. Нажмите **"Create API Key"**, дайте имя (например, `my-agent-key`) и скопируйте ключ.

**Ваш API-ключ будет выглядеть примерно так:**
```
ls-v2-7f8a9b1c2d3e4f5a6b7c8d9e0f1a2b3c
```

#### Шаг 2: Создание файла `.env`

Создайте в корне проекта файл `.env` со следующим содержимым:

```bash
# LangSmith Configuration
LANGCHAIN_TRACING_V2=true
LANGCHAIN_ENDPOINT="https://api.smith.langchain.com"
LANGCHAIN_API_KEY="ls-v2-ваш-ключ-сюда"
LANGCHAIN_PROJECT="my-rag-agent"

# Ollama Configuration (опционально)
OLLAMA_HOST="http://localhost:11434"
```

#### Шаг 3: Установка зависимостей

```bash
pip install python-dotenv
```

#### Шаг 4: Загрузка переменных в коде

В начале `monitoring.py` уже есть:

```python
from dotenv import load_dotenv
load_dotenv()
```

---

### 2.2. Полный код `monitoring.py`

```python
"""
monitoring.py - Агент на LangGraph с мониторингом через LangSmith
Лекция 6.5, Тема 2

Возможности:
- Интеграция с LangSmith для профессиональной трассировки
- Кастомное логирование в JSON-файлы
- Замер времени выполнения каждого шага
- Анализ медленных запросов
"""

import os
import math
import time
from dotenv import load_dotenv
from typing import Annotated, List, TypedDict

from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

# Подключаем кастомный логгер
from custom_logger import AgentLogger

# Загружаем переменные окружения (.env)
load_dotenv()

# Проверяем настройку LangSmith
if os.getenv("LANGCHAIN_API_KEY"):
    print("=" * 60)
    print("✅ LangSmith подключён")
    print(f"   Проект: {os.getenv('LANGCHAIN_PROJECT', 'default')}")
    print(f"   Трассировка: {os.getenv('LANGCHAIN_TRACING_V2', 'false')}")
    print("=" * 60)
else:
    print("ℹ️ LangSmith не настроен. Используется кастомное логирование.")
    print("   Для настройки создайте файл .env с LANGCHAIN_API_KEY")
    print("   Инструкция: https://smith.langchain.com/")
    print("=" * 60)

# Инициализируем кастомный логгер
logger = AgentLogger(log_dir="logs")

# ============================================================================
# 1. ИНСТРУМЕНТЫ
# ============================================================================

@tool
def calculate(expression: str) -> str:
    """
    Выполняет математические вычисления.
    Поддерживает: +, -, *, /, **, sqrt, pi, e.
    """
    try:
        safe_dict = {'sqrt': math.sqrt, 'pi': math.pi, 'e': math.e}
        result = eval(expression, {"__builtins__": {}}, safe_dict)
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка вычисления: {e}"

@tool
def get_current_time() -> str:
    """Возвращает текущие дату и время в формате ГГГГ-ММ-ДД ЧЧ:ММ:СС."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

tools = [calculate, get_current_time]

# ============================================================================
# 2. LLM С ИНСТРУМЕНТАМИ
# ============================================================================

SYSTEM_PROMPT = """
Ты — интеллектуальный агент с доступом к инструментам.

Инструменты:
- calculate: выполняет математические вычисления (пример: '2+2', 'sqrt(16)').
- get_current_time: возвращает текущие дату и время.

Правила:
1. Используй инструменты только когда это действительно нужно.
2. Если вопрос требует вычислений — используй calculate.
3. Если вопрос о времени/дате — используй get_current_time.
4. После получения результата, сформулируй чёткий ответ.
5. Не придумывай информацию, которой у тебя нет.
"""

llm = ChatOllama(model="qwen2.5:3b", temperature=0.0, num_predict=512)
llm_with_tools = llm.bind_tools(tools)

# ============================================================================
# 3. СОСТОЯНИЕ
# ============================================================================

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

# ============================================================================
# 4. УЗЛЫ ГРАФА
# ============================================================================

def agent_node(state: AgentState) -> AgentState:
    """Узел-агент: вызывает LLM с инструментами."""
    messages = state["messages"]
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages
    
    start_time = time.time()
    response = llm_with_tools.invoke(messages)
    duration = time.time() - start_time
    
    # Логируем вызов LLM
    logger.log_step({
        "step": "agent_decision",
        "session_id": "user_123",
        "duration": duration,
        "has_tool_calls": bool(hasattr(response, "tool_calls") and response.tool_calls),
        "tool_calls": [tc for tc in response.tool_calls] if hasattr(response, "tool_calls") and response.tool_calls else []
    })
    
    return {"messages": [response]}

def should_continue(state: AgentState) -> str:
    """Определяет, нужны ли инструменты или финальный ответ."""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        logger.log_step({
            "step": "routing",
            "session_id": "user_123",
            "decision": "tools",
            "tool_calls": [tc["name"] for tc in last_message.tool_calls]
        })
        return "tools"
    
    logger.log_step({
        "step": "routing",
        "session_id": "user_123",
        "decision": "final_answer"
    })
    return "final_answer"

def final_answer_node(state: AgentState) -> AgentState:
    """
    Генерирует финальный ответ на основе всей истории.
    """
    messages = state["messages"]
    
    # Добавляем системный промпт, если его нет
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages
    
    # ЯВНО ПРОСИМ МОДЕЛЬ СФОРМУЛИРОВАТЬ ФИНАЛЬНЫЙ ОТВЕТ
    # Это ключевое исправление — без этого модель может не дать ответ
    messages.append(HumanMessage(
        content="Сформулируй финальный ответ пользователю на основе всей доступной информации. Будь кратким и чётким."
    ))
    
    start_time = time.time()
    response = llm.invoke(messages)
    duration = time.time() - start_time
    
    # Если ответ всё равно пустой — используем последнее сообщение агента
    final_text = response.content
    if not final_text or len(final_text.strip()) < 2:
        # Ищем последнее сообщение от агента (без tool_calls)
        for msg in reversed(messages):
            if isinstance(msg, type(response)) and msg.content and msg.content.strip():
                final_text = msg.content
                break
        else:
            final_text = "Ответ сформирован, но модель не вернула текст."
    
    # Логируем финальный ответ
    logger.log_step({
        "step": "final_answer",
        "session_id": "user_123",
        "duration": duration,
        "answer": final_text[:500]
    })
    
    # Возвращаем сообщение с финальным ответом
    return {"messages": [HumanMessage(content=final_text)]}

tool_node = ToolNode(tools)

# ============================================================================
# 5. СБОРКА ГРАФА
# ============================================================================

builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_node("final_answer", final_answer_node)

builder.set_entry_point("agent")
builder.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", "final_answer": "final_answer"}
)
builder.add_edge("tools", "agent")
builder.add_edge("final_answer", END)

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

# ============================================================================
# 6. ЗАПУСК
# ============================================================================

if __name__ == "__main__":
    config = {"configurable": {"thread_id": "user_123"}, "recursion_limit": 10}
    
    questions = [
        "Сколько будет 2+2?",
        "Который сейчас час?"
    ]
    
    print("\n" + "=" * 60)
    print("🤖 АГЕНТ С МОНИТОРИНГОМ")
    print("=" * 60)
    
    for q in questions:
        print(f"\n{'=' * 60}")
        print(f"📝 Вопрос: {q}")
        print('-' * 60)
        
        # Логируем вопрос пользователя
        logger.log_step({
            "step": "user_query",
            "session_id": "user_123",
            "query": q
        })
        
        total_start = time.time()
        result = graph.invoke(
            {"messages": [("user", q)]},
            config=config
        )
        total_duration = time.time() - total_start
        
        last_msg = result["messages"][-1]
        answer_text = last_msg.content if last_msg.content else "пустой ответ"
        print(f"✅ Ответ: {answer_text}")
        print(f"⏱️ Общее время: {total_duration:.2f}с")
        
        # Логируем общее время
        logger.log_step({
            "step": "total",
            "session_id": "user_123",
            "total_duration": total_duration
        })
    
    print("\n" + "=" * 60)
    print(f"📁 Логи сохранены в папку: logs/")
    print("=" * 60)
```

---

### 2.3. Полный код `custom_logger.py`

```python
"""
custom_logger.py - Кастомное логирование для агента
Лекция 6.5, Тема 2

Записывает шаги агента в JSON-файл с временными метками.
Поддерживает:
- Логирование каждого шага
- Замер времени выполнения
- Сохранение в структурированном формате (JSON)
- Анализ медленных запросов
"""

import json
import logging
from datetime import datetime
from pathlib import Path
from typing import Dict, Any, Optional, List
from functools import wraps
import time


class AgentLogger:
    """
    Кастомный логгер для агента.
    
    Логи хранятся в папке logs/ в формате:
    agent_log_YYYY-MM-DD.json
    
    Каждая запись содержит:
    - timestamp: время записи
    - step: тип шага (user_query, agent_decision, tool_call, final_answer, total)
    - session_id: идентификатор сессии
    - дополнительные поля в зависимости от шага
    """
    
    def __init__(self, log_dir: str = "logs"):
        """
        Инициализация логгера.
        
        Аргументы:
            log_dir: папка для хранения логов
        """
        self.log_dir = Path(log_dir)
        self.log_dir.mkdir(exist_ok=True)
        
        # Настройка стандартного логирования для консоли
        logging.basicConfig(
            level=logging.INFO,
            format="%(asctime)s - %(levelname)s - %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S"
        )
        self.console_logger = logging.getLogger("AgentLogger")
    
    def log_step(self, step_data: Dict[str, Any]) -> None:
        """
        Записывает шаг в JSON-файл и выводит в консоль.
        
        Аргументы:
            step_data: словарь с данными шага
        """
        timestamp = datetime.now().isoformat()
        filename = self.log_dir / f"agent_log_{datetime.now().strftime('%Y-%m-%d')}.json"
        
        # Загружаем существующие записи
        existing = []
        if filename.exists():
            try:
                with open(filename, "r", encoding="utf-8") as f:
                    existing = json.load(f)
            except (json.JSONDecodeError, FileNotFoundError):
                existing = []
        
        # Добавляем новую запись
        entry = {
            "timestamp": timestamp,
            **step_data
        }
        existing.append(entry)
        
        # Сохраняем (с отступами для читаемости)
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(existing, f, ensure_ascii=False, indent=2)
        
        # Выводим краткую информацию в консоль
        self._print_console(step_data)
    
    def _print_console(self, step_data: Dict[str, Any]) -> None:
        """Выводит краткую информацию в консоль."""
        step = step_data.get("step", "unknown")
        
        if step == "user_query":
            self.console_logger.info(f"👤 Вопрос: {step_data.get('query', '')[:100]}")
        elif step == "agent_decision":
            has_tools = step_data.get("has_tool_calls", False)
            if has_tools:
                tools = step_data.get("tool_calls", [])
                tool_names = [t.get("name", "unknown") for t in tools]
                self.console_logger.info(f"🤖 Вызов инструментов: {', '.join(tool_names)}")
            else:
                self.console_logger.info("🤖 Агент решил ответить сразу")
        elif step == "tool_call":
            self.console_logger.info(f"🔧 Инструмент: {step_data.get('tool_name', 'unknown')}")
        elif step == "routing":
            self.console_logger.info(f"🔀 Маршрутизация: {step_data.get('decision', 'unknown')}")
        elif step == "final_answer":
            answer = step_data.get("answer", "")[:200]
            self.console_logger.info(f"💬 Ответ: {answer}...")
        elif step == "total":
            duration = step_data.get("total_duration", 0)
            self.console_logger.info(f"⏱️ Общее время: {duration:.2f}с")
    
    def log_user_query(self, query: str, session_id: str = "default") -> None:
        """Логирует вопрос пользователя."""
        self.log_step({
            "step": "user_query",
            "session_id": session_id,
            "query": query
        })
    
    def log_agent_decision(
        self,
        session_id: str,
        has_tool_calls: bool,
        tool_calls: Optional[list] = None,
        duration: Optional[float] = None
    ) -> None:
        """Логирует решение агента."""
        data = {
            "step": "agent_decision",
            "session_id": session_id,
            "has_tool_calls": has_tool_calls,
        }
        if tool_calls:
            data["tool_calls"] = tool_calls
        if duration is not None:
            data["duration"] = duration
        
        self.log_step(data)
    
    def log_routing(self, session_id: str, decision: str, tool_calls: Optional[list] = None) -> None:
        """Логирует маршрутизацию."""
        data = {
            "step": "routing",
            "session_id": session_id,
            "decision": decision,
        }
        if tool_calls:
            data["tool_calls"] = tool_calls
        self.log_step(data)
    
    def log_tool_call(
        self,
        tool_name: str,
        args: Dict,
        result: str,
        duration: float,
        session_id: str = "default"
    ) -> None:
        """Логирует вызов инструмента."""
        self.log_step({
            "step": "tool_call",
            "session_id": session_id,
            "tool_name": tool_name,
            "args": args,
            "result": result[:500],
            "duration": duration
        })
    
    def log_final_answer(self, answer: str, duration: float, session_id: str = "default") -> None:
        """Логирует финальный ответ."""
        self.log_step({
            "step": "final_answer",
            "session_id": session_id,
            "answer": answer[:500],
            "duration": duration
        })
    
    def log_total_time(self, duration: float, session_id: str = "default") -> None:
        """Логирует общее время выполнения запроса."""
        self.log_step({
            "step": "total",
            "session_id": session_id,
            "total_duration": duration
        })
    
    # ========================================================================
    # МЕТОДЫ ДЛЯ АНАЛИЗА ЛОГОВ
    # ========================================================================
    
    def get_logs(self, date: Optional[str] = None) -> List[Dict]:
        """
        Возвращает логи за указанную дату.
        
        Аргументы:
            date: дата в формате YYYY-MM-DD (если None — сегодня)
        
        Возвращает:
            Список записей лога
        """
        if date is None:
            date = datetime.now().strftime("%Y-%m-%d")
        
        filename = self.log_dir / f"agent_log_{date}.json"
        if not filename.exists():
            return []
        
        with open(filename, "r", encoding="utf-8") as f:
            return json.load(f)
    
    def get_slow_queries(self, threshold: float = 5.0) -> List[Dict]:
        """
        Находит медленные запросы.
        
        Аргументы:
            threshold: порог времени в секундах
        
        Возвращает:
            Список медленных запросов
        """
        slow_queries = []
        for log_file in self.log_dir.glob("agent_log_*.json"):
            with open(log_file, "r", encoding="utf-8") as f:
                entries = json.load(f)
            
            # Находим все запросы и их общее время
            for entry in entries:
                if entry.get("step") == "total":
                    duration = entry.get("total_duration", 0)
                    if duration > threshold:
                        # Находим соответствующий вопрос
                        query = "unknown"
                        timestamp = entry.get("timestamp", "")
                        for e in entries:
                            if e.get("step") == "user_query" and e.get("timestamp") < timestamp:
                                query = e.get("query", "unknown")
                                break
                        slow_queries.append({
                            "timestamp": timestamp,
                            "query": query,
                            "duration": duration
                        })
        
        return sorted(slow_queries, key=lambda x: x["duration"], reverse=True)
    
    def get_routing_errors(self) -> List[Dict]:
        """
        Находит ошибки маршрутизации.
        
        Возвращает:
            Список записей с ошибками
        """
        errors = []
        for log_file in self.log_dir.glob("agent_log_*.json"):
            with open(log_file, "r", encoding="utf-8") as f:
                entries = json.load(f)
            
            for entry in entries:
                if entry.get("step") == "routing":
                    decision = entry.get("decision", "")
                    if decision not in ["tools", "final_answer"]:
                        errors.append(entry)
        
        return errors


# ============================================================================
# ДЕКОРАТОР ДЛЯ ЗАМЕРА ВРЕМЕНИ
# ============================================================================

def timer(func):
    """
    Декоратор для замера времени выполнения функции.
    
    Использование:
        @timer
        def my_function():
            return result
    
        result, duration = my_function()
    """
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        duration = time.time() - start
        return result, duration
    return wrapper


# ============================================================================
# ПРИМЕР ИСПОЛЬЗОВАНИЯ
# ============================================================================

if __name__ == "__main__":
    # Создаём логгер
    logger = AgentLogger()
    
    # Логируем тестовые данные
    logger.log_user_query("Сколько будет 2+2?", session_id="test_123")
    logger.log_agent_decision(
        "test_123",
        has_tool_calls=True,
        tool_calls=[{"name": "calculate", "args": {"expression": "2+2"}}],
        duration=0.5
    )
    logger.log_tool_call(
        "calculate",
        {"expression": "2+2"},
        "Результат: 4",
        0.1,
        "test_123"
    )
    logger.log_routing("test_123", "tools", tool_calls=["calculate"])
    logger.log_final_answer("2+2 равно 4", 0.3, "test_123")
    logger.log_total_time(0.9, "test_123")
    
    print("\n" + "=" * 60)
    print("✅ Тестовые логи сохранены в папку logs/")
    print(f"📁 Путь: {logger.log_dir.absolute()}")
    print("=" * 60)
    
    # Демонстрация анализа
    slow = logger.get_slow_queries(threshold=0.5)
    if slow:
        print(f"\n🐌 Медленные запросы (>0.5с):")
        for q in slow:
            print(f"   {q['query']} - {q['duration']:.2f}с")
```

---

### 2.4. Ожидаемый вывод при запуске

**Запуск мониторинга:**

```bash
python monitoring.py
```

**Вывод:**

```
============================================================
✅ LangSmith подключён
   Проект: my-rag-agent
   Трассировка: true
============================================================

============================================================
🤖 АГЕНТ С МОНИТОРИНГОМ
============================================================

============================================================
📝 Вопрос: Сколько будет 2+2?
------------------------------------------------------------
2026-08-06 19:30:45 - INFO - 👤 Вопрос: Сколько будет 2+2?
2026-08-06 19:30:51 - INFO - 🤖 Вызов инструментов: calculate
2026-08-06 19:30:51 - INFO - 🔀 Маршрутизация: tools
2026-08-06 19:30:51 - INFO - 🤖 Агент решил ответить сразу
2026-08-06 19:30:51 - INFO - 🔀 Маршрутизация: final_answer
2026-08-06 19:30:52 - INFO - 💬 Ответ: 2+2 равно 4...
✅ Ответ: 2+2 равно 4
⏱️ Общее время: 6.55с
2026-08-06 19:30:52 - INFO - ⏱️ Общее время: 6.55с

============================================================
📝 Вопрос: Который сейчас час?
------------------------------------------------------------
2026-08-06 19:30:52 - INFO - 👤 Вопрос: Который сейчас час?
2026-08-06 19:30:53 - INFO - 🤖 Вызов инструментов: get_current_time
2026-08-06 19:30:53 - INFO - 🔀 Маршрутизация: tools
2026-08-06 19:30:54 - INFO - 🤖 Агент решил ответить сразу
2026-08-06 19:30:54 - INFO - 🔀 Маршрутизация: final_answer
2026-08-06 19:30:55 - INFO - 💬 Ответ: Текущее время: 2026-08-06 19:30:55...
✅ Ответ: Текущее время: 2026-08-06 19:30:55
⏱️ Общее время: 2.65с
2026-08-06 19:30:55 - INFO - ⏱️ Общее время: 2.65с

============================================================
📁 Логи сохранены в папку: logs/
============================================================
```

---

### 2.5. Как анализировать логи

Логи — это не просто запись происходящего, а **инструмент для улучшения системы**.

#### 2.5.1. Поиск медленных запросов

```python
from custom_logger import AgentLogger

logger = AgentLogger()
slow = logger.get_slow_queries(threshold=5.0)

if slow:
    print("🐌 Медленные запросы (>5с):")
    for q in slow:
        print(f"   {q['query']} - {q['duration']:.2f}с")
else:
    print("✅ Медленных запросов нет")
```

#### 2.5.2. Анализ ошибок маршрутизации

```python
from custom_logger import AgentLogger

logger = AgentLogger()
errors = logger.get_routing_errors()

if errors:
    print("⚠️ Ошибки маршрутизации:")
    for e in errors:
        print(f"   {e}")
```

#### 2.5.3. Просмотр логов за день

```python
from custom_logger import AgentLogger
import json

logger = AgentLogger()
logs = logger.get_logs()

print(f"📊 Логов за сегодня: {len(logs)}")
for log in logs[:5]:  # первые 5
    print(json.dumps(log, ensure_ascii=False, indent=2))
```

---

### 2.6. Практические советы по мониторингу

| Что делать | Как часто | Зачем |
|------------|-----------|-------|
| **Просматривать логи** | Раз в день | Быстро замечать ошибки |
| **Анализировать медленные запросы** | Раз в неделю | Оптимизировать узкие места |
| **Проверять ошибки маршрутизации** | Раз в неделю | Улучшать промпты |
| **Обновлять промпты** | По необходимости | Улучшать качество ответов |
| **Настраивать параметры поиска** | Раз в месяц | Адаптировать под новые данные |

---

## Краткий итог Тема 2

- **LangSmith** предоставляет профессиональную трассировку с дашбордом, где видно каждый шаг агента. Бесплатный план доступен для всех.
- **API-ключ** LangSmith создаётся в настройках аккаунта и добавляется в файл `.env` вместе с переменными `LANGCHAIN_TRACING_V2=true` и `LANGCHAIN_PROJECT`.
- **Кастомное логирование** (`custom_logger.py`) — альтернатива для случаев, когда LangSmith недоступен или нужна полная приватность данных.
- **Ключевое исправление:** в `final_answer_node` добавлен явный запрос к модели на формулировку финального ответа, что решает проблему с пустыми ответами.
- **Логи помогают:**
  - Находить медленные запросы.
  - Отлавливать ошибки маршрутизации.
  - Анализировать качество поиска.
  - Улучшать систему на основе реальных данных.

Теперь агент стал прозрачным. Пора ускорить его, чтобы он мог обслуживать много пользователей одновременно.

## Домашнее задание к лекции 6.5

В этой лекции мы превратили прототип агента в промышленную систему: улучшили поиск через гибридный подход и реранкинг, настроили мониторинг и логирование (LangSmith + кастомное), обсудили ускорение, интерфейсы, безопасность и самооценку. Ваша задача — применить эти улучшения на практике, провести сравнительные эксперименты, проанализировать логи и предложить оптимизации.

> **Важно:** Все задания выполняются на основе вашего собственного кода из предыдущих лекций (например, агент с инструментами из Лекции 6.4 или multi‑agent из Лекции 6.6). Вы можете модифицировать скрипты `advanced_search.py`, `monitoring.py`, `custom_logger.py` и интегрировать их в своего агента. Для экспериментов используйте локальную модель Ollama (например, `qwen2.5:3b`).

---

### Обязательная часть (5 баллов)

#### 1. Сравнение обычного семантического поиска и улучшенного поиска (гибрид + реранкинг) (2 балла)

В лекции мы показали, что гибридный поиск (BM25 + эмбеддинги) с последующим реранкингом через кросс‑энкодер значительно повышает точность на запросах с артикулами, датами и редкими терминами. Ваша задача — проверить это на своих данных.

**Что сделать:**

1. Интегрируйте функции из `advanced_search.py` (гибридный поиск, реранкинг) в вашего агента (например, в узел-исследователя или в ретривер). Убедитесь, что BM25-индекс строится на основе ваших чанков, а кросс‑энкодер загружается при инициализации.

2. Подготовьте **набор из 15 вопросов** к вашим документам, включающих:
   - 5 вопросов с **точными терминами** (артикулы, коды, даты, имена собственные).
   - 5 вопросов с **естественно‑языковыми запросами** (например, «Как работает RAG?»).
   - 5 вопросов, требующих **смешанного поиска** (термин + контекст).

3. Для каждого вопроса запустите агента **дважды**:
   - **Версия A**: только семантический поиск (Chroma, как раньше).
   - **Версия B**: улучшенный поиск (гибрид + реранкинг).

   Зафиксируйте для каждого запуска:
   - Топ‑3 найденных чанка (их тексты и оценки).
   - Финальный ответ агента.
   - Время выполнения поисковой части.

4. **В отчёте**:
   - Приведите таблицу сравнения по всем вопросам: для каждого вопроса укажите, какие чанки нашла каждая версия и насколько релевантным был финальный ответ (оцените вручную по шкале 1–5).
   - Рассчитайте **среднюю точность** (доля вопросов, где топ‑1 чанк релевантен) для обеих версий.
   - Проанализируйте, на каких типах вопросов улучшенный поиск дал наибольший прирост качества, а на каких — не дал существенного улучшения.
   - Сделайте вывод, стоит ли использовать гибридный поиск + реранкинг для ваших данных, и оцените приемлемость дополнительного времени выполнения.

---

#### 2. Настройка мониторинга и анализ логов (3 балла)

В лекции мы добавили два уровня мониторинга: LangSmith (профессиональная трассировка) и кастомное логирование в JSON-файлы. Ваша задача — настроить мониторинг для своего агента, собрать логи и провести их анализ.

**Что сделать:**

1. **Настройте LangSmith** (если у вас есть доступ):
   - Создайте аккаунт на [LangSmith](https://smith.langchain.com/), получите API-ключ.
   - Добавьте переменные окружения в файл `.env` (как в лекции).
   - Запустите своего агента на **10–15 вопросах** и убедитесь, что трассировки появляются в дашборде.

2. **Внедрите кастомное логирование** (класс `AgentLogger` из `custom_logger.py`):
   - Добавьте вызовы `logger.log_step()` в ключевые узлы агента (принятие решения, вызов инструментов, маршрутизация, финальный ответ).
   - Запустите агента на тех же вопросах, чтобы логи сохранились в папку `logs/`.

3. **Проанализируйте собранные логи** (как из LangSmith, так и из JSON-файлов):
   - Найдите **3 самых медленных запроса** (порог > 5 секунд). Для каждого определите, какой шаг занял больше всего времени (LLM, поиск, инструменты).
   - Найдите **ошибки маршрутизации** (например, агент выбрал инструмент, хотя должен был ответить сразу, или наоборот).
   - Найдите **запросы, где финальный ответ был неполным или неточным**, и проследите по логам, почему это произошло (например, плохие чанки, неправильный инструмент).

4. **В отчёте**:
   - Приведите фрагменты логов (из LangSmith или JSON) для одного медленного запроса, одной ошибки маршрутизации и одного некачественного ответа, с комментариями.
   - На основе анализа предложите **конкретные улучшения**:
     - Как изменить промпты, чтобы улучшить маршрутизацию?
     - Как оптимизировать поиск (например, увеличить `top_k`, изменить параметры чанкинга)?
     - Как ускорить работу (кеширование, асинхронность)?
   - Оцените, насколько мониторинг помог вам понять поведение системы, и дайте рекомендацию, стоит ли использовать LangSmith в продакшене.

---

### Дополнительная часть (эксперименты — выполните минимум 3 из 6, каждый до 2 баллов)

#### 3. Кеширование повторяющихся запросов (2 балла)

В реальных системах пользователи часто задают одни и те же вопросы. Кеширование ответов экономит время и ресурсы.

**Что сделать:**

1. Реализуйте простой кеш для вашего агента:
   - Используйте словарь в памяти или `functools.lru_cache`.
   - Ключом кеша может быть хеш вопроса (или сам вопрос).
   - При повторном вопросе возвращайте сохранённый ответ, не вызывая LLM и поиск.

2. Протестируйте на **5 повторяющихся вопросах** (задайте каждый вопрос дважды). Замерьте время первого и второго ответа.

3. **В отчёте**:
   - Опишите, как вы реализовали кеш (с каким TTL, с учётом ли сессии).
   - Приведите таблицу с временем выполнения для первого и второго запроса.
   - Обсудите, какие вопросы безопасно кешировать, а какие — нет (например, вопросы о текущем времени).
   - Оцените, даёт ли кеширование ощутимый выигрыш для вашего сценария.

---

#### 4. Веб‑интерфейс на Streamlit (2 балла)

Сделайте свой агент доступным через простой веб‑интерфейс, чтобы им могли пользоваться не только разработчики.

**Что сделать:**

1. Установите Streamlit: `pip install streamlit`.
2. Создайте файл `app.py`, который:
   - Загружает вашего агента (инициализирует LLM, ретривер, инструменты).
   - Показывает поле ввода вопроса и кнопку отправки.
   - Выводит ответ агента и, опционально, найденные чанки.
   - Поддерживает историю диалога (в рамках сессии).
3. Запустите приложение и проверьте на **5 вопросах**.

4. **В отчёте**:
   - Приведите скриншоты интерфейса.
   - Опишите, как вы организовали передачу состояния между запросами.
   - Обсудите, какие сложности возникли при интеграции агента в веб‑приложение (например, управление памятью, долгие ответы).

---

#### 5. Асинхронность и параллелизм (2 балла)

Чтобы повысить пропускную способность, можно переделать агента на асинхронные вызовы.

**Что сделать:**

1. Используя `async`/`await` и методы `ainvoke` в LangChain/LangGraph, реализуйте асинхронную версию вашего агента.
2. Напишите функцию, которая параллельно обрабатывает **10 вопросов** с помощью `asyncio.gather()`.
3. Замерьте общее время последовательного и параллельного выполнения.

4. **В отчёте**:
   - Приведите код асинхронной версии.
   - Покажите сравнение времени (последовательно vs параллельно).
   - Обсудите, насколько эффективен параллелизм для ваших вопросов (зависит ли от сложности?).
   - Оцените, есть ли ограничения (например, лимиты модели Ollama, количество одновременных запросов).

---

#### 6. Безопасность: валидация ввода и контроль действий (2 балла)

Агент может выполнять опасные действия (например, удалять файлы или отправлять письма), если мы дадим ему такие инструменты. В реальных системах нужен контроль.

**Что сделать:**

1. Добавьте в агента **валидацию ввода**:
   - Проверяйте, что вопрос не содержит вредоносных команд (SQL-инъекции, системные вызовы).
   - Ограничьте длину вопроса.

2. Реализуйте **механизм подтверждения** для «опасных» инструментов:
   - Определите список опасных инструментов (например, `delete_file`, `send_email`).
   - Перед их выполнением запрашивайте подтверждение у пользователя (через консоль или интерфейс).

3. Протестируйте на **трёх сценариях**: безопасный вопрос, вопрос с подозрительным содержимым, вопрос, требующий опасного инструмента.

4. **В отчёте**:
   - Опишите, как вы реализовали валидацию и подтверждение.
   - Приведите примеры и покажите, как система реагирует.
   - Обсудите, насколько такие меры достаточны для продакшена, и предложите дополнительные (например, ролевой доступ).

---

#### 7. Самооценка и улучшение ответа (2 балла)

В лекции мы упомянули самооценку (Reflexion) — агент оценивает свой ответ и при необходимости улучшает его. Реализуйте простую версию.

**Что сделать:**

1. Добавьте узел **«оценщик»** после генерации финального ответа:
   - Оценщик — это LLM, которая получает вопрос, ответ и выставляет оценку (1–10) с комментарием.
   - Если оценка ниже 7 (или другого порога), запускается узел **«улучшатель»**, который генерирует новый ответ с учётом замечаний.
   - Цикл повторяется не более 2 раз.

2. Протестируйте на **5 вопросах**, где ответ может быть неполным. Зафиксируйте, улучшился ли ответ после самооценки.

3. **В отчёте**:
   - Опишите архитектуру и промпты для оценщика и улучшателя.
   - Приведите примеры: первый ответ, оценка, замечания, улучшенный ответ.
   - Оцените, насколько самооценка повышает качество и не слишком ли замедляет работу.

---

#### 8. Упаковка в Docker (2 балла)

Чтобы развернуть агента на сервере, удобно использовать Docker.

**Что сделать:**

1. Напишите `Dockerfile`, который:
   - Использует базовый образ с Python 3.11.
   - Устанавливает зависимости (`langchain`, `langgraph`, `ollama`, `chromadb` и др.).
   - Копирует ваш код.
   - Указывает команду запуска (например, `streamlit run app.py` или `python main.py`).

2. Соберите образ и запустите контейнер. Убедитесь, что агент работает внутри контейнера и доступен по порту.

3. **В отчёте**:
   - Приведите содержимое `Dockerfile` и инструкцию по сборке/запуску.
   - Опишите, какие трудности возникли (например, подключение к Ollama внутри контейнера).
   - Обсудите, какие переменные окружения нужно передавать (API-ключи, настройки модели).

---

### Формат сдачи

1. **Код** всех модификаций сохраните в отдельной папке (или используйте Git). Код должен быть хорошо задокументирован и легко запускаться.

2. **Отчёт** в формате `.md` или `.pdf`, содержащий:
   - Для **каждого выполненного задания**:
     - Краткое описание, что сделано.
     - Ключевые результаты (таблицы, графики, скриншоты, примеры ответов, фрагменты логов).
     - Ваши выводы и обоснования.
   - Для обязательной части — чёткие сравнения и численные оценки.
   - Для дополнительной части — достаточные детали, чтобы можно было воспроизвести эксперимент.

3. **Лог-файлы** (по желанию) — приложите фрагменты логов для подтверждения экспериментов (особенно для задания 2).

---

### Критерии оценки

| Раздел | Максимум баллов |
|--------|----------------|
| **Обязательная часть** | 5 |
| Задание 1 (сравнение поисков) | 2 |
| Задание 2 (мониторинг и анализ) | 3 |
| **Дополнительная часть** (каждый пункт) | до 2 (максимум +10) |
| За каждый выполненный пункт: 1 балл за реализацию, 1 балл за анализ/выводы | |
| **Итого** | 15 |

**Штрафы:**
- Отсутствие отчёта или невнятные выводы — минус 2 балла.
- Код без комментариев и с плохим форматированием — минус 1 балл.
- Использование готовых ответов без собственного анализа — минус 2 балла.

---

### Рекомендации

- Начинайте с обязательной части — она даёт базовое понимание улучшений для продакшена.
- Для экспериментов используйте **один и тот же набор вопросов**, чтобы результаты были сопоставимы.
- Вносите изменения в код аккуратно, делайте резервные копии.
- Активно используйте логирование — оно поможет понять, что происходит внутри системы.
- В выводах опирайтесь не только на численные данные, но и на качественный анализ ответов.
- Если у вас нет доступа к LangSmith, используйте кастомное логирование — оно даёт аналогичную информацию для анализа.